In [61]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import adi
import binascii
import time

In [62]:
def verify_crc(received_frame):
    received_payload = received_frame[:-16]
    received_crc = received_frame[-16:]
    byte_data = np.packbits(received_payload)  
    crc = binascii.crc_hqx(byte_data, 0xFFFF)  
    expected_crc = np.array(list(np.binary_repr(crc, width=16)), dtype=np.uint8)  
    if np.array_equal(received_crc, expected_crc):
        return True  
    else:
        return False 

def binary_array_to_int(binary_array):
    return int("".join(map(str, binary_array)), 2)

def bin_to_text(binary_str):
    decoded_bits = "".join(str(bit) for bit in binary_str)
    text = ""
    for i in range(0, len(decoded_bits), 7):
        byte = decoded_bits[i:i+7]
        if len(byte) == 7 :
            text += chr(int(byte, 2))
    return text

In [63]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch
import string
from sentence_transformers import SentenceTransformer, util
import re
from collections import Counter
import numpy as np

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
model11 = SentenceTransformer('all-MiniLM-L6-v2')

In [64]:
with open('test_file_1.txt') as f:
# with open('Datasets/sherlock_holmes/novels/The Hound of the Baskervilles.txt') as f:
  test_text = f.read()
text_words = re.findall(r"\w+|[^\w\s]", test_text, re.UNICODE)

no_seed = 50
seed_text = " ".join(text_words[:no_seed])

In [65]:
seed_text

'In the digital age , privacy has become one of the most debated and critical issues . With every click , swipe , and search , individuals leave behind a trail of data . This data , often collected without explicit consent , is used by corporations , advertisers ,'

In [66]:
def fine_freq_sync(signal_name):
    samples = signal_name
    N = len(samples)
    phase = 0
    freq = 0
    alpha = 0.132
    beta = 0.00932
    fine_correct = np.zeros(N, dtype=np.complex64)
    freq_log = []
    for i in range(N):
        fine_correct[i] = samples[i] * np.exp(-1j*phase)
        error = np.real(fine_correct[i]) * np.imag(fine_correct[i]) 
        freq += (beta * error)
        freq_log.append(freq * fs / (2*np.pi)) 
        phase += freq + (alpha * error)
        while phase >= 2*np.pi:
            phase -= 2*np.pi
        while phase < 0:
            phase += 2*np.pi
    return fine_correct


def Time_synchronization(signal_name):
    samples  = signal_name[100:]
    samples_interpolated = signal.resample_poly(samples, 16, 1)
    mu = 0 
    out = np.zeros(len(samples) + 10, dtype=np.complex64)
    out_rail = np.zeros(len(samples) + 10, dtype=np.complex64) #
    i_in = 0 
    i_out = 2 
    while i_out < len(samples) and i_in+16 < len(samples):
        out[i_out] = samples_interpolated[i_in*16 + int(mu*16)]
        out_rail[i_out] = int(np.real(out[i_out]) > 0) + 1j*int(np.imag(out[i_out]) > 0)
        x = (out_rail[i_out] - out_rail[i_out-2]) * np.conj(out[i_out-1])
        y = (out[i_out] - out[i_out-2]) * np.conj(out_rail[i_out-1])
        mm_val = np.real(y - x)
        mu += sps + 0.3*mm_val
        i_in += int(np.floor(mu))
        mu = mu - np.floor(mu) 
        i_out += 1 
    out = out[2:i_out] 
    return out

In [67]:
def Receiver_Model(seed_text, rx_text):
    rx_words = re.findall(r"\w+|[^\w\s]", rx_text, re.UNICODE)
    rx_collect = []
    for k in range(len(rx_words)):
        actual = rx_words[k]
        original_tokens = seed_text.split()
        # if actual != '_':
        #     pred1 = actual
        #     new_prompt = ' '.join(original_tokens[1:] + [pred1.strip()])
        #     rx_collect.append(pred1)
        # else:
        #     input_ids = tokenizer.encode(seed_text, return_tensors='pt')
        #     with torch.no_grad():
        #         output = model.generate(
        #             input_ids,
        #             max_new_tokens=1,
        #             do_sample=False,
        #             pad_token_id=tokenizer.eos_token_id)
        #     predicted_token = output[0][input_ids.shape[-1]:]
        #     pred = tokenizer.decode(predicted_token, skip_special_tokens=True).strip()
        #     new_prompt = ' '.join(original_tokens[1:] + [pred.strip()])
        #     rx_collect.append(' {'+ pred+'} ')
        if actual != '_':
            pred1 = actual
            new_prompt = ' '.join(original_tokens[1:] + [pred1.strip()])
            rx_collect.append(pred1)
        else:
            input_ids = tokenizer.encode(seed_text, return_tensors='pt')
            with torch.no_grad():
                output = model.generate(
                    input_ids,
                    max_new_tokens=1,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id)

            predicted_token = output[0][input_ids.shape[-1]:]
            pred = tokenizer.decode(predicted_token, skip_special_tokens=True).strip()
            new_prompt = ' '.join(original_tokens[1:] + [pred.strip()])

            rx_collect.append(f' <span style="color:red;">{pred}</span> ')
        seed_text = new_prompt
    rx_decoded = ' '.join(rx_collect)
    return rx_decoded, seed_text

In [68]:
def cut_before_star_partition(s: str) -> str:
    before, sep, after = s.partition('*')
    return before

from IPython.display import display, HTML

def show_message(text):
    display(HTML(f"""
    <div style="
        background-color:white;
        color:black;
        font-size:24px;
        padding:20px;
        border:2px solid black;
        border-radius:10px;
        width:300px;
        text-align:center;">
        {text}
    </div>
    """))


In [69]:
prev_num = 1000
no_seed = 50
seed_text = " ".join(text_words[:no_seed])

In [70]:
while(1):    
    flag = 0
    while(flag == 0):
        sample_rate = 10e6 # Hz
        carrier_freq = 985e6 # Hz
        num_samps = 100000 

        sdr2 = adi.Pluto("ip:192.168.3.1")
        sdr2.sample_rate = int(sample_rate)

        sdr2.rx_lo = int(carrier_freq)
        sdr2.rx_rf_bandwidth = int(sample_rate)
        sdr2.rx_buffer_size = num_samps
        sdr2.gain_control_mode_chan0 = 'slow_attack'
        # sdr2.rx_hardwaregain_chan0 = 70 # dB

        rx_samples = sdr2.rx()
        variance = np.var(rx_samples.real)

        if variance > 50000:

            sps = 8
            num_taps = 101
            beta = 0.34
            Ts = sps 
            t = np.arange(num_taps) - (num_taps-1)//2
            rc_pulse = np.sinc(t/Ts) * np.cos(np.pi*beta*t/Ts) / (1 - (2*beta*t/Ts)**2)
            matched_filter =  rc_pulse
            MF_op = np.convolve(rx_samples, matched_filter, mode='same')

            max_allowed = 1.5
            rx_max = np.max(np.abs(MF_op))
            scale_coarse = max_allowed / rx_max
            normalized_rx = MF_op * scale_coarse

            squared_FO = normalized_rx**2
            fs = sample_rate
            psd = np.fft.fftshift(np.abs(np.fft.fft(squared_FO)))
            f = np.linspace(-fs/2.0, fs/2.0, len(psd))
            max_freq = f[np.argmax(psd)]

            Ts = 1/fs
            t = np.arange(0, Ts*len(normalized_rx), Ts) # create time vector
            coarse_corrected = normalized_rx * np.exp(-1j*2*np.pi*max_freq*t/2.0)

            # Unit power normalization
            power = np.mean(np.abs(coarse_corrected)**2)
            normalized_2 = coarse_corrected / np.sqrt(power)

            fine_corrected = fine_freq_sync(normalized_2)

            time_sync = Time_synchronization(fine_corrected)

            barker_code = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
            barker_correlated = np.correlate(time_sync, barker_code, mode='full')
            offset  = np.argmax(np.abs(barker_correlated))+1

            peak = np.real(barker_correlated)[offset-1]
            if peak < 0:
                time_sync = -1*time_sync

            payload_size = 1015
            n_bits = payload_size + 16 + 10

            fr_sync_op = time_sync[offset:offset+n_bits]
            
            recovered_bits = (fr_sync_op > 0).astype(int)
            
            received_frame = recovered_bits

            if verify_crc(received_frame):
                num = binary_array_to_int(received_frame[:10])
                if num != prev_num:
                    print('\ncrc pass:', num)
                    flag = 1
                    payload_bits = received_frame[10:payload_size+10]
                    decoded_text = bin_to_text(payload_bits)
                    # print('\nReceived text: ', decoded_text, end = '')

                    in_string = cut_before_star_partition(decoded_text)
                    # print('\nprev num, num:', prev_num, num)
                
                    
                    
                    final_decoded, seed_rx_next = Receiver_Model(seed_text, in_string)
                    seed_text = seed_rx_next
                    prev_num = num
                    # print('\nDecoded final: ', final_decoded)
                    show_message(final_decoded)
            else:
                print('\n crc fail -----------')  
                      


crc pass: 0



crc pass: 1



 crc fail -----------

crc pass: 2



 crc fail -----------

crc pass: 3



 crc fail -----------

crc pass: 4


...


 crc fail -----------

crc pass: 9



crc pass: 10



crc pass: 11



 crc fail -----------

crc pass: 12


KeyboardInterrupt: 